# 1. Introducción

**Problema industrial:** Análisis de vida útil de rodamientos en operación continua.

**Activo analizado:** Rodamientos BRG-001 a BRG-050 de bombas de molienda.

**Origen de datos:** Registro de horas hasta falla desde mantenimiento y PI (simulado).

**Objetivo del análisis:** Ajustar distribución Weibull y estimar parámetros β (forma) y η (escala).


# 2. Carga de librerías

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LAB_DIR = Path.cwd()
os.chdir(LAB_DIR)
OUTPUT_DIR = LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
EXCEL_DIR = LAB_DIR / "excel"
DATA_PATH = LAB_DIR / "data" / "datos_exportados_PI.csv"
from scipy import stats
from scipy.special import gamma as gamma_fn


# 3. Lectura de datos PI System

Simulamos una exportación del historiador PI con columnas: `Timestamp`, `Tag`, `Value`, `Unit`, `Quality`.

In [ ]:
df_pi = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
print(f"Registros cargados: {len(df_pi):,}")
df_pi.head(10)


# 4. Exploración del dato

In [ ]:
print("Columnas:", df_pi.columns.tolist())
print("\nEstadísticas por tag:")
display(df_pi.groupby("Tag")["Value"].describe())

calidad = df_pi["Quality"].value_counts(normalize=True) * 100
print("\nCalidad del dato (%):")
print(calidad.round(2))

faltantes = df_pi["Value"].isna().sum()
print(f"\nValores faltantes: {faltantes}")

df_good = df_pi[df_pi["Quality"] == "GOOD"].copy()
tendencia = df_good.groupby("Tag")["Value"].agg(["mean", "std", "min", "max"])
print("\nTendencia central por tag:")
display(tendencia)


# 5. Análisis matemático

Ajuste Weibull de dos parámetros a vidas útil.

In [ ]:
vidas = pd.read_excel(EXCEL_DIR / "modelo_ingenieria.xlsx", sheet_name="Vidas_Weibull")
t = vidas["Vida_horas"].values

shape, loc, scale = stats.weibull_min.fit(t, floc=0)
print(f"β (forma) = {shape:.3f}")
print(f"η (escala) = {scale:.1f} horas")

t_medio = scale * gamma_fn(1 + 1 / shape)
resultados_export = pd.DataFrame({
    "Parametro": ["Beta_forma", "Eta_escala_h", "Vida_media_h", "N_muestras"],
    "Valor": [shape, scale, t_medio, len(t)],
})
display(resultados_export)


# 6. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(t, bins=15, density=True, alpha=0.7, color="steelblue", label="Datos")
x = np.linspace(0, t.max() * 1.2, 200)
axes[0].plot(x, stats.weibull_min.pdf(x, shape, loc=0, scale=scale), "r-", lw=2, label="Weibull ajustada")
axes[0].set_xlabel("Vida útil (h)")
axes[0].set_ylabel("Densidad")
axes[0].set_title("Histograma y PDF Weibull")
axes[0].legend()

axes[1].plot(x, stats.weibull_min.sf(x, shape, loc=0, scale=scale), color="darkgreen", lw=2)
axes[1].set_xlabel("Tiempo (h)")
axes[1].set_ylabel("Confiabilidad R(t)")
axes[1].set_title("Curva de confiabilidad")
axes[1].grid(True, alpha=0.3)


# 7. Exportación

In [ ]:
resultados_path = OUTPUT_DIR / "resultado_analisis.csv"
graficos_path = OUTPUT_DIR / "graficos.png"
excel_resultado = EXCEL_DIR / "modelo_resultado.xlsx"

resultados_export.to_csv(resultados_path, index=False)
with pd.ExcelWriter(excel_resultado, engine="openpyxl") as writer:
    resultados_export.to_excel(writer, sheet_name="Weibull", index=False)
    vidas.to_excel(writer, sheet_name="Vidas", index=False)

plt.tight_layout()
plt.savefig(graficos_path, dpi=150, bbox_inches="tight")
print(f"CSV exportado: {resultados_path}")
print(f"Gráficos exportados: {graficos_path}")
print(f"Excel exportado: {excel_resultado}")


# 8. Interpretación ingenieril

## Interpretación para mantenimiento

β > 1 indica fallas por desgaste. Programar reemplazo preventivo antes de η para maximizar disponibilidad sin incurrir en cambios prematuros.
